# Analyse des Entreprises Belges — BCE/KBO
### Travaux Pratiques — Data Platform

> **Données** : BCE Open Data — Snapshot du 27 juin 2026 (Extract n°404, Version 1.0.0)

| Entreprise | Numéro BCE | Secteur | Date création |
|---|---|---|---|
| **Google Belgium** | 0878.065.378 | Services informatiques (NACE 62900) | 21-12-2005 |
| **Apple Retail Belgium** | 0836.157.420 | Commerce high-tech (NACE 47400) | 06-05-2011 |
| **SNCB** | 0203.430.576 | Transport ferroviaire (NACE 49110) | 01-01-1968 |

---

**Structure du notebook** :
1. Section 0 — Configuration & Chargement des données CSV KBO
2. Section 1 — Données brutes : entités complètes depuis les CSVs
3. Section 2 — Informations enrichies (KBO public, NBB, Notaire, eJustice)
4. Section 3 — Analyse financière & Visualisations (2021–2025)
5. Glossaire des indicateurs financiers


## Section 0 — Configuration et Chargement des Données

Imports, constantes et chargement des 7 fichiers CSV de la BCE.

In [ ]:
# ─── Librairies ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display, HTML
import warnings, os, time
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)

# ─── Chemin des données KBO ───────────────────────────────────────────────────
DATA_PATH = '/workspaces/Data-Platform/TP1/data/'

# ─── Entreprises cibles (numéros BCE sans points) ────────────────────────────
COMPANIES = {
    'Google Belgium':       '0878065378',
    'Apple Retail Belgium': '0836157420',
    'SNCB':                 '0203430576',
}

# Le site KBO public utilise le numéro SANS zéro initial
KBO_NUMBERS = {
    'Google Belgium':       '878065378',
    'Apple Retail Belgium': '836157420',
    'SNCB':                 '203430576',
}

# ─── En-têtes HTTP pour les requêtes web ─────────────────────────────────────
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'fr-BE,fr;q=0.9',
}

print("✅ Configuration chargée.")
print(f"   Chemin données : {DATA_PATH}")
print(f"   Entreprises    : {list(COMPANIES.keys())}")


### Chargement des fichiers CSV BCE Open Data

In [ ]:
# Chargement des 7 fichiers CSV de la BCE
# dtype=str : préserve les zéros initiaux des numéros (ex: '0878065378')
print("Chargement en cours...")

enterprise_df    = pd.read_csv(f'{DATA_PATH}enterprise.csv',    dtype=str)
denomination_df  = pd.read_csv(f'{DATA_PATH}denomination.csv',  dtype=str)
address_df       = pd.read_csv(f'{DATA_PATH}address.csv',       dtype=str)
activity_df      = pd.read_csv(f'{DATA_PATH}activity.csv',      dtype=str)
contact_df       = pd.read_csv(f'{DATA_PATH}contact.csv',       dtype=str)
establishment_df = pd.read_csv(f'{DATA_PATH}establishment.csv', dtype=str)
code_df          = pd.read_csv(f'{DATA_PATH}code.csv',          dtype=str)
meta_df          = pd.read_csv(f'{DATA_PATH}meta.csv',          dtype=str)

# Normaliser les colonnes de numéros BCE (enlever les points éventuels)
for df, cols in [
    (enterprise_df,    ['EnterpriseNumber']),
    (denomination_df,  ['EntityNumber']),
    (address_df,       ['EntityNumber']),
    (activity_df,      ['EntityNumber']),
    (contact_df,       ['EntityNumber']),
    (establishment_df, ['EstablishmentNumber', 'EnterpriseNumber']),
]:
    for col in cols:
        if col in df.columns:
            df[col] = df[col].str.replace('.', '', regex=False).str.strip()

print("\n✅ Données chargées :")
datasets = [
    ('enterprise',    enterprise_df),
    ('denomination',  denomination_df),
    ('address',       address_df),
    ('activity',      activity_df),
    ('contact',       contact_df),
    ('establishment', establishment_df),
    ('code',          code_df),
]
for name, df in datasets:
    print(f"   {name:15s} → {len(df):>10,} lignes")

print("\n📅 Métadonnées du snapshot BCE :")
display(meta_df)


### Fonctions utilitaires

In [ ]:
# ─── Normalisation ───────────────────────────────────────────────────────────
def normalize_number(n):
    '''Supprime les points et espaces d un numéro BCE.'''
    return str(n).replace('.', '').replace(' ', '').strip()

def format_bce(n):
    '''Formate XXXXXXXXXX → XXXX.XXX.XXX (format d affichage officiel BCE).'''
    n = normalize_number(n)
    return f"{n[:4]}.{n[4:7]}.{n[7:]}" if len(n) == 10 else n

# ─── Lookup code KBO → libellé ────────────────────────────────────────────────
def get_code_description(category, code_val, lang='1'):
    '''Traduit un code KBO en libellé via code.csv (lang: 1=FR, 2=NL).'''
    mask = (
        (code_df['Category'] == category) &
        (code_df['Code'] == str(code_val).strip()) &
        (code_df['Language'] == str(lang))
    )
    res = code_df[mask]['Description']
    return res.values[0] if len(res) > 0 else f"[{code_val}]"

# ─── Récupération du nom officiel ─────────────────────────────────────────────
def get_denomination(entity_number, lang='1', type_code=None):
    '''Retourne le nom officiel d une entité (lang: 1=FR, 2=NL).'''
    n = normalize_number(entity_number)
    mask = (
        (denomination_df['EntityNumber'] == n) &
        (denomination_df['Language'] == lang)
    )
    if type_code:
        mask = mask & (denomination_df['TypeOfDenomination'] == type_code)
    res = denomination_df[mask]['Denomination']
    if len(res) > 0:
        return res.values[0]
    # Fallback sans filtre sur le type de dénomination
    fallback = denomination_df[
        (denomination_df['EntityNumber'] == n) &
        (denomination_df['Language'] == lang)
    ]['Denomination']
    return fallback.values[0] if len(fallback) > 0 else 'N/A'

# ─── Récupération de l adresse ────────────────────────────────────────────────
def get_address_str(entity_number):
    '''Retourne l adresse formatée du siège social (TypeOfAddress=REGO).'''
    n = normalize_number(entity_number)
    df = address_df[address_df['EntityNumber'] == n]
    rego = df[df['TypeOfAddress'] == 'REGO']
    row = rego.iloc[0] if not rego.empty else (df.iloc[0] if not df.empty else None)
    if row is None:
        return 'N/A'
    street  = (str(row.get('StreetFR', '') or row.get('StreetNL', '') or '')).strip()
    num     = str(row.get('HouseNumber', '') or '').strip()
    box_val = str(row.get('Box', '') or '').strip()
    zipcode = str(row.get('Zipcode', '') or '').strip()
    city    = (str(row.get('MunicipalityFR', '') or row.get('MunicipalityNL', '') or '')).strip()
    street_full = f"{street} {num}".strip()
    if box_val and box_val not in ('nan', ''):
        street_full += f" bte {box_val}"
    parts = [p for p in [street_full, f"{zipcode} {city}".strip()] if p]
    return ', '.join(parts) if parts else 'N/A'

print("✅ Fonctions utilitaires définies.")


---
## Section 1 — Entités de Base (Données CSV KBO)

Consolidation des données brutes depuis les 7 CSVs pour chaque entreprise.
Source : BCE Open Data — Snapshot du 27 juin 2026.


In [ ]:
def get_entity(numero):
    '''
    Consolide toutes les données KBO d une entreprise à partir des CSVs.
    Returns: dict avec enterprise, denominations, address, activities, contacts, establishments.
    '''
    n = normalize_number(numero)
    ent = enterprise_df[enterprise_df['EnterpriseNumber'] == n]
    if ent.empty:
        print(f"⚠️  Entreprise {numero} introuvable dans enterprise.csv")
        return None
    return {
        'numero':         n,
        'enterprise':     ent.iloc[0].to_dict(),
        'denominations':  denomination_df[denomination_df['EntityNumber'] == n],
        'address':        address_df[address_df['EntityNumber'] == n],
        'activities':     activity_df[activity_df['EntityNumber'] == n],
        'contacts':       contact_df[contact_df['EntityNumber'] == n],
        'establishments': establishment_df[establishment_df['EnterpriseNumber'] == n],
    }

def afficher_entity(entity, verbose=True):
    '''
    Affiche une fiche entreprise lisible avec codes traduits.
    '''
    if entity is None:
        return
    e   = entity['enterprise']
    n   = entity['numero']
    nom = get_denomination(n, lang='1') or get_denomination(n, lang='2')

    statut   = get_code_description('Status',              e.get('Status', ''))
    sit_jur  = get_code_description('JuridicalSituation',  e.get('JuridicalSituation', ''))
    form_jur = get_code_description('JuridicalForm',       e.get('JuridicalForm', ''))
    type_ent = get_code_description('TypeOfEnterprise',    e.get('TypeOfEnterprise', ''))
    adresse  = get_address_str(n)

    print('=' * 70)
    print(f"  {nom.upper()}  ({format_bce(n)})")
    print('=' * 70)
    print(f"  Statut              : {statut}")
    print(f"  Situation juridique : {sit_jur}")
    print(f"  Forme juridique     : {form_jur}")
    print(f"  Type d entreprise   : {type_ent}")
    print(f"  Date de création    : {e.get('StartDate', 'N/A')}")
    print(f"  Adresse             : {adresse}")

    # Activités NACE groupées par version
    acts = entity['activities']
    if not acts.empty:
        print("  Activités NACE :")
        for version in sorted(acts['NaceVersion'].unique(), reverse=True):
            print(f"    Version {version} :")
            for _, row in acts[acts['NaceVersion'] == version].iterrows():
                classif = get_code_description('Classification', row['Classification'])
                group   = get_code_description('ActivityGroup',  row['ActivityGroup'])
                print(f"      {row['NaceCode']:7s}  [{classif}]  {group}")

    phones   = entity['contacts'][entity['contacts']['ContactType'] == 'TEL']['Value'].tolist()
    emails   = entity['contacts'][entity['contacts']['ContactType'] == 'EMAIL']['Value'].tolist()
    websites = entity['contacts'][entity['contacts']['ContactType'] == 'WEB']['Value'].tolist()
    print(f"  Téléphone           : {phones or ['—']}")
    print(f"  Email               : {emails or ['—']}")
    print(f"  Site web            : {websites or ['—']}")
    print(f"  Établissements      : {len(entity['establishments'])}")

    if verbose:
        print("\n  Dénominations :")
        print(entity['denominations'][['Language','TypeOfDenomination','Denomination']].to_string(index=False))
    print()


In [ ]:
# ─── Affichage des 3 entreprises depuis les CSVs ─────────────────────────────
for name, numero in COMPANIES.items():
    entity = get_entity(numero)
    afficher_entity(entity)


---
## Section 2 — Informations Enrichies

| Source | Sections |
|---|---|
| **CSV KBO** | 2.1, 2.2, 2.3, 2.8, 2.10 |
| **KBO Public** (`kbopub.economie.fgov.be`) | 2.4, 2.5 |
| **Notaire.be** (`statuts.notaire.be`) | 2.6 |
| **NBB/CBSO** (`consult.cbso.nbb.be`) | 2.7 |
| **eJustice** (`ejustice.just.fgov.be`) | 2.9 |


---
### 2.1 — Informations Générales

Nom officiel, numéro BCE, numéro TVA, adresse, activité principale, date de création.
Source : `denomination.csv` + `enterprise.csv` + `address.csv` + `activity.csv`


In [ ]:
def get_general_info(numero):
    '''
    Retourne un dict avec les informations générales d une entreprise.
    Source : enterprise.csv + denomination.csv + address.csv + activity.csv
    '''
    n = normalize_number(numero)
    ent = enterprise_df[enterprise_df['EnterpriseNumber'] == n]
    if ent.empty:
        return None
    row = ent.iloc[0]

    nom_fr = get_denomination(n, lang='1', type_code='001')
    nom_nl = get_denomination(n, lang='2', type_code='001')
    if nom_fr == 'N/A':
        nom_fr = get_denomination(n, lang='1')
    if nom_nl == 'N/A':
        nom_nl = get_denomination(n, lang='2')

    # Activité principale : classification MAIN, version NACE la plus récente
    acts = activity_df[
        (activity_df['EntityNumber'] == n) &
        (activity_df['Classification'] == 'MAIN')
    ].sort_values('NaceVersion', ascending=False)

    if not acts.empty:
        main_act  = acts.iloc[0]
        nace_code = main_act['NaceCode']
        nace_grp  = get_code_description('ActivityGroup', main_act['ActivityGroup'])
    else:
        nace_code = 'N/A'
        nace_grp  = 'N/A'

    return {
        'Dénomination (FR)':   nom_fr,
        'Dénomination (NL)':   nom_nl,
        'Numéro BCE':          format_bce(n),
        'Numéro TVA':          f"BE {format_bce(n)}",
        'Adresse (siège)':     get_address_str(n),
        'Activité principale': f"{nace_code} — {nace_grp}",
        'Date de création':    row.get('StartDate', 'N/A'),
        'Statut':              get_code_description('Status', row.get('Status', '')),
    }

for name, numero in COMPANIES.items():
    info = get_general_info(numero)
    print(f"\n{'=' * 65}")
    print(f"  {name}")
    print(f"{'=' * 65}")
    if info:
        for k, v in info.items():
            print(f"  {k:<25}: {v}")


---
### 2.2 — Informations Juridiques

Forme juridique, situation juridique, type d'entreprise, numéro TVA.
Source : `enterprise.csv` + `code.csv`


In [ ]:
def get_legal_info(numero):
    '''
    Retourne les informations juridiques d une entreprise depuis les CSVs.
    Source : enterprise.csv + code.csv
    '''
    n = normalize_number(numero)
    ent = enterprise_df[enterprise_df['EnterpriseNumber'] == n]
    if ent.empty:
        return None
    row = ent.iloc[0]
    return {
        'Numéro BCE':          format_bce(n),
        'Numéro TVA':          f"BE {format_bce(n)}",
        'Forme juridique':     get_code_description('JuridicalForm',      row.get('JuridicalForm', '')),
        'Situation juridique': get_code_description('JuridicalSituation', row.get('JuridicalSituation', '')),
        'Type d entreprise':   get_code_description('TypeOfEnterprise',   row.get('TypeOfEnterprise', '')),
        'Statut':              get_code_description('Status',             row.get('Status', '')),
        'Date de création':    row.get('StartDate', 'N/A'),
    }

legal_data = {name: get_legal_info(num) for name, num in COMPANIES.items()}
df_legal = pd.DataFrame(legal_data)
print("\n📋 Informations Juridiques — Tableau comparatif")
display(df_legal)


---
### 2.3 — Activités NACE

Toutes les activités enregistrées (codes NACE 2003, 2008, 2025) avec leur classification.
Source : `activity.csv` + `code.csv`


In [ ]:
def get_activities(numero):
    '''
    Retourne toutes les activités NACE d une entreprise avec leurs libellés.
    Source : activity.csv + code.csv
    '''
    n = normalize_number(numero)
    acts = activity_df[activity_df['EntityNumber'] == n].copy()
    if acts.empty:
        return pd.DataFrame()

    acts['Classification_label'] = acts['Classification'].apply(
        lambda x: get_code_description('Classification', x)
    )
    acts['ActivityGroup_label'] = acts['ActivityGroup'].apply(
        lambda x: get_code_description('ActivityGroup', x)
    )
    return acts[[
        'NaceVersion', 'NaceCode', 'Classification', 'Classification_label',
        'ActivityGroup', 'ActivityGroup_label'
    ]].sort_values(['NaceVersion', 'Classification'], ascending=[False, True]).reset_index(drop=True)

for name, numero in COMPANIES.items():
    print(f"\n{'=' * 65}")
    print(f"  {name}  ({format_bce(numero)})")
    print(f"{'=' * 65}")
    acts = get_activities(numero)
    if not acts.empty:
        display(acts)
    else:
        print("  Aucune activité trouvée.")


---
### 2.4 — Dirigeants et Représentants

Scraping du site KBO public pour récupérer la liste des mandataires.

**URL** : `https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer={numero}`

> Le site KBO public utilise le numéro **sans zéro initial** (ex: `878065378`).


In [ ]:
def scrape_directors(kbo_number, delay=2):
    '''
    Scrape la liste des dirigeants/mandataires depuis le site KBO public.
    kbo_number : numéro sans zéro initial (ex: 878065378)
    '''
    url = (
        f"https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html"
        f"?ondernemingsnummer={kbo_number}"
    )
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        dirigeants = []

        # Le tableau des fonctions est précédé d un en-tête contenant 'fonct' ou 'mandataire'
        for heading in soup.find_all(['h2', 'h3', 'caption', 'th', 'div', 'td']):
            txt = heading.get_text(strip=True).lower()
            if any(kw in txt for kw in ['fonct', 'mandataire', 'bestuurder', 'kwaliteit']):
                table = heading.find_next('table')
                if table:
                    for tr in table.find_all('tr')[1:]:
                        tds = tr.find_all('td')
                        if len(tds) >= 1:
                            nom     = tds[0].get_text(strip=True)
                            qualite = tds[1].get_text(strip=True) if len(tds) > 1 else ''
                            depuis  = tds[2].get_text(strip=True) if len(tds) > 2 else ''
                            if nom:
                                dirigeants.append({
                                    'Nom / Entité': nom,
                                    'Qualité':      qualite,
                                    'Depuis':       depuis,
                                })
                    break

        time.sleep(delay)
        return (pd.DataFrame(dirigeants) if dirigeants
                else pd.DataFrame(columns=['Nom / Entité', 'Qualité', 'Depuis']))

    except Exception as e:
        print(f"   ⚠️  Erreur scraping KBO dirigeants : {e}")
        return pd.DataFrame(columns=['Nom / Entité', 'Qualité', 'Depuis'])

print("Récupération des dirigeants (site KBO public)...")
for name, kbo_num in KBO_NUMBERS.items():
    print(f"\n{'=' * 65}")
    print(f"  {name}  (KBO n°: {kbo_num})")
    print(f"{'=' * 65}")
    df_dir = scrape_directors(kbo_num)
    if not df_dir.empty:
        display(df_dir.reset_index(drop=True))
    else:
        print("  Section non récupérée automatiquement.")
        print(f"  → Consulter : https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer={kbo_num}")


---
### 2.5 — Liens entre Entités

Entités liées (filiales, groupe, participations).
Source : site KBO public (même URL que section 2.4).


In [ ]:
def scrape_entity_links(kbo_number, delay=1):
    '''
    Scrape les liens entre entités depuis le site KBO public.
    kbo_number : numéro sans zéro initial
    '''
    url = (
        f"https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html"
        f"?ondernemingsnummer={kbo_number}"
    )
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        liens = []

        for heading in soup.find_all(['h2', 'h3', 'caption', 'th', 'div', 'td']):
            txt = heading.get_text(strip=True).lower()
            if any(kw in txt for kw in ['lien', 'link', 'verbinding', 'relat']):
                table = heading.find_next('table')
                if table:
                    for tr in table.find_all('tr')[1:]:
                        tds = tr.find_all('td')
                        if len(tds) >= 2:
                            liens.append({
                                'Entité liée':  tds[0].get_text(strip=True),
                                'Type de lien': tds[1].get_text(strip=True),
                                'Depuis':       tds[2].get_text(strip=True) if len(tds) > 2 else '',
                            })
                    break

        time.sleep(delay)
        return (pd.DataFrame(liens) if liens
                else pd.DataFrame(columns=['Entité liée', 'Type de lien', 'Depuis']))

    except Exception as e:
        print(f"   ⚠️  Erreur scraping liens : {e}")
        return pd.DataFrame(columns=['Entité liée', 'Type de lien', 'Depuis'])

print("Récupération des liens entre entités (site KBO public)...")
for name, kbo_num in KBO_NUMBERS.items():
    print(f"\n{'=' * 65}")
    print(f"  {name}")
    print(f"{'=' * 65}")
    df_liens = scrape_entity_links(kbo_num)
    if not df_liens.empty:
        display(df_liens.reset_index(drop=True))
    else:
        print("  Aucun lien trouvé ou section non disponible.")
        print(f"  → Consulter : https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer={kbo_num}")


---
### 2.6 — Documents Juridiques (Statuts)

Statuts officiels déposés auprès des notaires belges.
Source : `https://statuts.notaire.be/stapor_v1/enterprise/{numero}/statutes`


In [ ]:
def scrape_statutes(enterprise_number, delay=2):
    '''
    Récupère la liste des statuts notariaux via l API statuts.notaire.be.
    enterprise_number : numéro BCE complet (avec zéro initial, 10 chiffres)
    '''
    n   = normalize_number(enterprise_number)
    url = f"https://statuts.notaire.be/stapor_v1/enterprise/{n}/statutes"

    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code == 200:
            try:
                data  = resp.json()
                items = data if isinstance(data, list) else data.get('statutes', data.get('items', []))
                statutes = []
                for item in items:
                    statutes.append({
                        'Date':        item.get('date', item.get('depositDate', 'N/A')),
                        'Type':        item.get('type', item.get('documentType', 'N/A')),
                        'Description': item.get('description', item.get('title', 'N/A')),
                        'URL':         item.get('url', item.get('documentUrl', '')),
                    })
                time.sleep(delay)
                return pd.DataFrame(statutes)
            except Exception:
                time.sleep(delay)
                return pd.DataFrame()
        else:
            print(f"   → Statut HTTP {resp.status_code}")
            return pd.DataFrame()
    except Exception as e:
        print(f"   ⚠️  Erreur statuts.notaire.be : {e}")
        return pd.DataFrame()

print("Récupération des statuts notariaux...")
for name, numero in COMPANIES.items():
    n = normalize_number(numero)
    print(f"\n{'=' * 65}")
    print(f"  {name}  ({format_bce(numero)})")
    print(f"{'=' * 65}")
    df_stat = scrape_statutes(numero)
    if df_stat is not None and not df_stat.empty:
        display(df_stat.reset_index(drop=True))
    else:
        print("  Statuts non récupérés automatiquement.")
        print(f"  → URL directe : https://statuts.notaire.be/stapor_v1/enterprise/{n}/statutes")


---
### 2.7 — Comptes Annuels (NBB/CBSO)

Dépôts des comptes annuels à la Banque Nationale de Belgique.

**URL** : `https://consult.cbso.nbb.be/consult-enterprise/{numero}`

**Stratégie de stockage** :
- PDFs (depuis 2000) → `/data/nbb/pdfs/{numero}/{annee}.pdf`
- CSVs (depuis 2021) → `/data/nbb/csvs/{numero}/{annee}.csv`
- Exclure les comptes **consolidés** (code modèle commençant par `mc`)
- Dédupliquer : **1 dépôt par année** (préférer FR sur NL)


In [ ]:
def get_annual_accounts_nbb(enterprise_number, download=False, base_path=None):
    '''
    Récupère la liste des comptes annuels depuis la NBB/CBSO.
    Optionnellement télécharge les PDFs et CSVs si download=True.
    Exclut les comptes consolidés (mc-*) et déduplique par année (préférence FR).
    '''
    n     = normalize_number(enterprise_number)
    n_fmt = format_bce(n)

    filings = []

    # ─── Tentative via l API REST NBB ────────────────────────────────────────
    api_url = f"https://consult.cbso.nbb.be/api/institutions/{n_fmt}/accounts"
    try:
        resp = requests.get(api_url, headers=HEADERS, timeout=20)
        if resp.status_code == 200:
            data  = resp.json()
            items = data if isinstance(data, list) else data.get('accounts', data.get('items', []))
            for item in items:
                model_code = item.get('modelCode', '')
                # Exclure les comptes consolidés
                if str(model_code).lower().startswith('mc'):
                    continue
                year      = str(item.get('exerciseYear', item.get('year', '')))
                lang      = item.get('language', item.get('lang', ''))
                dep_date  = item.get('depositDate', '')
                pdf_url   = item.get('pdfUrl', '')
                csv_url   = item.get('csvUrl', '')
                filings.append({
                    'Année':      year,
                    'Langue':     lang,
                    'Modèle':     model_code,
                    'Date dépôt': dep_date,
                    'PDF':        '✅' if pdf_url else '—',
                    'CSV':        '✅' if csv_url else '—',
                    '_pdf_url':   pdf_url,
                    '_csv_url':   csv_url,
                })
    except Exception as e:
        print(f"   API NBB non disponible : {e}")

    # ─── Fallback : scraping page web CBSO ───────────────────────────────────
    if not filings:
        cbso_url = f"https://consult.cbso.nbb.be/consult-enterprise/{n}"
        try:
            resp2 = requests.get(cbso_url, headers=HEADERS, timeout=20)
            if resp2.status_code == 200:
                soup = BeautifulSoup(resp2.text, 'html.parser')
                for table in soup.find_all('table'):
                    rows = table.find_all('tr')
                    for tr in rows[1:]:
                        tds = tr.find_all('td')
                        if len(tds) >= 3:
                            model_code = tds[3].get_text(strip=True) if len(tds) > 3 else ''
                            if str(model_code).lower().startswith('mc'):
                                continue
                            filings.append({
                                'Année':      tds[0].get_text(strip=True),
                                'Langue':     tds[1].get_text(strip=True),
                                'Date dépôt': tds[2].get_text(strip=True),
                                'Modèle':     model_code,
                                'PDF':        '✅',
                                'CSV':        '—',
                                '_pdf_url':   '',
                                '_csv_url':   '',
                            })
                    if filings:
                        break
        except Exception as e2:
            print(f"   ⚠️  Scraping CBSO non disponible : {e2}")

    if not filings:
        return pd.DataFrame()

    df = pd.DataFrame(filings)

    # Dédupliquer : 1 dépôt par année, préférer FR
    if 'Année' in df.columns and 'Langue' in df.columns:
        df_dedup = (df
            .sort_values('Langue', ascending=True)  # FR avant NL alphabétiquement
            .drop_duplicates(subset=['Année'], keep='first')
            .sort_values('Année', ascending=False)
            .reset_index(drop=True)
        )
    else:
        df_dedup = df.reset_index(drop=True)

    # ─── Téléchargement optionnel ─────────────────────────────────────────────
    if download and base_path:
        os.makedirs(f"{base_path}/pdfs/{n}", exist_ok=True)
        os.makedirs(f"{base_path}/csvs/{n}", exist_ok=True)
        for _, row in df_dedup.iterrows():
            year = row.get('Année', '')
            # PDF (depuis 2000)
            if row.get('_pdf_url') and year:
                try:
                    r = requests.get(row['_pdf_url'], headers=HEADERS, timeout=30)
                    with open(f"{base_path}/pdfs/{n}/{year}.pdf", 'wb') as fout:
                        fout.write(r.content)
                    print(f"   PDF {year} ✅ téléchargé.")
                except Exception as ep:
                    print(f"   PDF {year} ❌ : {ep}")
            # CSV (depuis 2021)
            if row.get('_csv_url') and year:
                try:
                    y_int = int(year)
                except ValueError:
                    y_int = 0
                if y_int >= 2021:
                    try:
                        r = requests.get(row['_csv_url'], headers=HEADERS, timeout=30)
                        with open(f"{base_path}/csvs/{n}/{year}.csv", 'wb') as fout:
                            fout.write(r.content)
                        print(f"   CSV {year} ✅ téléchargé.")
                    except Exception as ec:
                        print(f"   CSV {year} ❌ : {ec}")

    return df_dedup[['Année', 'Langue', 'Modèle', 'Date dépôt', 'PDF', 'CSV']]

print("Récupération des comptes annuels (NBB/CBSO)...")
for name, numero in COMPANIES.items():
    n = normalize_number(numero)
    print(f"\n{'=' * 65}")
    print(f"  {name}  ({format_bce(numero)})")
    print(f"{'=' * 65}")
    df_ca = get_annual_accounts_nbb(numero)
    if df_ca is not None and not df_ca.empty:
        display(df_ca)
    else:
        print("  Données non récupérées automatiquement.")
        print(f"  → URL directe : https://consult.cbso.nbb.be/consult-enterprise/{n}")

# Pour télécharger les fichiers, appeler avec download=True :
# df = get_annual_accounts_nbb('0878065378', download=True, base_path='/workspaces/Data-Platform/TP1/data/nbb')


---
### 2.8 — Établissements

Liste de tous les établissements (unités d'exploitation) de chaque entreprise.
Source : `establishment.csv` + `address.csv`


In [ ]:
def get_establishments(enterprise_number):
    '''
    Retourne la liste des établissements avec adresses.
    Source : establishment.csv + address.csv
    '''
    n     = normalize_number(enterprise_number)
    estbs = establishment_df[establishment_df['EnterpriseNumber'] == n].copy()
    if estbs.empty:
        return pd.DataFrame()

    results = []
    for _, row in estbs.iterrows():
        est_num = row['EstablishmentNumber']
        results.append({
            'Numéro établissement': est_num,
            'Adresse':              get_address_str(est_num),
            'Date création':        row.get('StartDate', 'N/A'),
        })

    return (pd.DataFrame(results)
            .sort_values('Date création', ascending=False)
            .reset_index(drop=True))

for name, numero in COMPANIES.items():
    estbs = get_establishments(numero)
    nb    = len(estbs)
    print(f"\n{'=' * 65}")
    print(f"  {name}  ({format_bce(numero)})  — {nb} établissement(s)")
    print(f"{'=' * 65}")
    if not estbs.empty:
        # Afficher max 20 lignes pour les entreprises avec beaucoup d établissements
        display(estbs.head(20))
        if nb > 20:
            print(f"  ... et {nb - 20} autres établissements.")
    else:
        print("  Aucun établissement trouvé.")


---
### 2.9 — Publications (eJustice)

Publications légales au Moniteur belge / Belgisch Staatsblad.
Source : `https://www.ejustice.just.fgov.be/cgi_tsv/list.pl?language=fr&btw={tva}&page=1`


In [ ]:
def scrape_ejustice(enterprise_number, max_pages=3, delay=2):
    '''
    Récupère les publications légales depuis eJustice (Moniteur belge).
    enterprise_number : numéro BCE avec zéro initial (10 chiffres)
    '''
    n   = normalize_number(enterprise_number)
    publications = []

    for page in range(1, max_pages + 1):
        url = (
            f"https://www.ejustice.just.fgov.be/cgi_tsv/list.pl"
            f"?language=fr&btw={n}&page={page}"
        )
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            if resp.status_code != 200:
                break
            soup = BeautifulSoup(resp.text, 'html.parser')
            found = 0
            for table in soup.find_all('table'):
                rows = table.find_all('tr')
                for tr in rows[1:]:
                    tds = tr.find_all('td')
                    if len(tds) >= 3:
                        date_txt = tds[0].get_text(strip=True)
                        # Filtre minimal : la date doit contenir un chiffre
                        if not any(c.isdigit() for c in date_txt):
                            continue
                        numac   = tds[1].get_text(strip=True) if len(tds) > 1 else ''
                        type_p  = tds[2].get_text(strip=True) if len(tds) > 2 else ''
                        lnk_tag = tr.find('a', href=True)
                        lnk     = lnk_tag['href'] if lnk_tag else ''
                        if lnk and lnk.startswith('/'):
                            lnk = f"https://www.ejustice.just.fgov.be{lnk}"
                        publications.append({
                            'Date':  date_txt,
                            'NUMAC': numac,
                            'Type':  type_p,
                            'Lien':  lnk,
                        })
                        found += 1
            if found == 0:
                break
            time.sleep(delay)
        except Exception as e:
            print(f"   ⚠️  eJustice page {page} : {e}")
            break

    return (pd.DataFrame(publications) if publications
            else pd.DataFrame(columns=['Date', 'NUMAC', 'Type', 'Lien']))

print("Récupération des publications eJustice...")
for name, numero in COMPANIES.items():
    n = normalize_number(numero)
    print(f"\n{'=' * 65}")
    print(f"  {name}  ({format_bce(numero)})")
    print(f"{'=' * 65}")
    df_pub = scrape_ejustice(numero)
    if not df_pub.empty:
        print(f"  {len(df_pub)} publication(s) trouvée(s) :")
        display(df_pub.head(10).reset_index(drop=True))
    else:
        print("  Aucune publication trouvée automatiquement.")
        print(f"  → URL directe : https://www.ejustice.just.fgov.be/cgi_tsv/list.pl?language=fr&btw={n}&page=1")


---
### 2.10 — Informations de Contact

Coordonnées (téléphone, fax, email, site web) et adresse du siège.
Source : `contact.csv` + `address.csv`


In [ ]:
def get_contact_info(enterprise_number):
    '''
    Retourne les coordonnées de contact depuis les CSVs KBO.
    Source : contact.csv + address.csv
    '''
    n        = normalize_number(enterprise_number)
    contacts = contact_df[contact_df['EntityNumber'] == n]

    phones   = contacts[contacts['ContactType'] == 'TEL']['Value'].tolist()
    fax      = contacts[contacts['ContactType'] == 'FAX']['Value'].tolist()
    emails   = contacts[contacts['ContactType'] == 'EMAIL']['Value'].tolist()
    websites = contacts[contacts['ContactType'] == 'WEB']['Value'].tolist()

    return {
        'Téléphone(s)': ', '.join(phones)   if phones   else '—',
        'Fax':          ', '.join(fax)      if fax      else '—',
        'Email(s)':     ', '.join(emails)   if emails   else '—',
        'Site(s) web':  ', '.join(websites) if websites else '—',
        'Adresse':      get_address_str(n),
    }

contact_data = {name: get_contact_info(num) for name, num in COMPANIES.items()}
df_contacts  = pd.DataFrame(contact_data)
print("\n📋 Informations de Contact — Tableau comparatif")
display(df_contacts)


---
## Section 3 — Analyse Financière (2021–2025)

Indicateurs calculés d'après le **Plan Comptable Minimum Normalisé (PCMN)** belge.
Données issues des comptes annuels déposés à la NBB/CBSO.


### 3.1 — Tableau des Indicateurs Financiers

In [ ]:
# ─── Google Belgium — Données réelles NBB/CBSO ───────────────────────────────
# Source : comptes annuels déposés à la Banque Nationale (extraits du notebook de référence)
google_fin = pd.DataFrame({
    'chiffre_affaires': [63_066_577,  77_386_505,  94_095_721,  92_512_761,  95_789_805],
    'marge_brute':      [63_066_577,  77_386_505,  94_095_721,  92_512_761,  95_789_805],
    'ebitda':           [10_155_360,  14_155_232,  16_530_148,  16_373_468,  16_461_765],
    'resultat_net':     [ 6_529_995,   9_532_020,  12_471_274,  12_985_558,  12_374_735],
    'tresorerie':       [         0,           0,           0,           0,           0],
    'dettes_fin':       [         0,           0,           0,           0,           0],
    'fonds_propres':    [30_438_332,  39_970_352,  52_441_626,  65_427_185,  77_801_920],
}, index=[2021, 2022, 2023, 2024, 2025])
google_fin.index.name = 'Année'

# ─── Apple Retail Belgium — Estimations indicatives ──────────────────────────
# ⚠️  À remplacer par les données réelles extraites via la Section 2.7 (NBB/CBSO)
apple_fin = pd.DataFrame({
    'chiffre_affaires': [320_000_000, 380_000_000, 430_000_000, 460_000_000, 490_000_000],
    'marge_brute':      [140_000_000, 165_000_000, 190_000_000, 200_000_000, 215_000_000],
    'ebitda':           [ 35_000_000,  42_000_000,  52_000_000,  55_000_000,  60_000_000],
    'resultat_net':     [ 18_000_000,  22_000_000,  28_000_000,  30_000_000,  33_000_000],
    'tresorerie':       [ 50_000_000,  60_000_000,  70_000_000,  75_000_000,  80_000_000],
    'dettes_fin':       [ 40_000_000,  40_000_000,  35_000_000,  30_000_000,  25_000_000],
    'fonds_propres':    [ 80_000_000,  95_000_000, 115_000_000, 135_000_000, 158_000_000],
}, index=[2021, 2022, 2023, 2024, 2025])
apple_fin.index.name = 'Année'

# ─── SNCB — Estimations indicatives ──────────────────────────────────────────
# ⚠️  À remplacer par les données réelles extraites via la Section 2.7 (NBB/CBSO)
sncb_fin = pd.DataFrame({
    'chiffre_affaires': [3_700_000_000, 3_850_000_000, 4_050_000_000, 4_200_000_000, 4_350_000_000],
    'marge_brute':      [1_100_000_000, 1_200_000_000, 1_300_000_000, 1_380_000_000, 1_450_000_000],
    'ebitda':           [  380_000_000,   420_000_000,   460_000_000,   490_000_000,   520_000_000],
    'resultat_net':     [   50_000_000,    70_000_000,    90_000_000,   110_000_000,   120_000_000],
    'tresorerie':       [  500_000_000,   550_000_000,   600_000_000,   650_000_000,   700_000_000],
    'dettes_fin':       [2_800_000_000, 2_750_000_000, 2_700_000_000, 2_650_000_000, 2_600_000_000],
    'fonds_propres':    [3_000_000_000, 3_050_000_000, 3_120_000_000, 3_200_000_000, 3_290_000_000],
}, index=[2021, 2022, 2023, 2024, 2025])
sncb_fin.index.name = 'Année'

# ─── Calcul des ratios pour les 3 entreprises ─────────────────────────────────
companies_fin = {
    'Google Belgium':       google_fin,
    'Apple Retail Belgium': apple_fin,
    'SNCB':                 sncb_fin,
}
for df_fin in companies_fin.values():
    df_fin['taux_marge_brute_%'] = (df_fin['marge_brute']  / df_fin['chiffre_affaires'] * 100).round(1)
    df_fin['taux_ebitda_%']      = (df_fin['ebitda']        / df_fin['chiffre_affaires'] * 100).round(1)
    df_fin['marge_nette_%']      = (df_fin['resultat_net']  / df_fin['chiffre_affaires'] * 100).round(1)
    df_fin['croissance_ca_%']    = df_fin['chiffre_affaires'].pct_change().mul(100).round(1)
    df_fin['dette_nette']        = df_fin['dettes_fin'] - df_fin['tresorerie']

# Affichage des tableaux
fmt = {
    'chiffre_affaires':   '{:,.0f} €',
    'marge_brute':        '{:,.0f} €',
    'ebitda':             '{:,.0f} €',
    'resultat_net':       '{:,.0f} €',
    'fonds_propres':      '{:,.0f} €',
    'taux_marge_brute_%': '{:.1f}%',
    'taux_ebitda_%':      '{:.1f}%',
    'marge_nette_%':      '{:.1f}%',
    'croissance_ca_%':    '{:.1f}%',
}
cols = ['chiffre_affaires', 'marge_brute', 'ebitda', 'resultat_net',
        'fonds_propres', 'taux_marge_brute_%', 'taux_ebitda_%',
        'marge_nette_%', 'croissance_ca_%']

for name, df_fin in companies_fin.items():
    print(f"\n{'=' * 70}")
    print(f"  {name}  — Indicateurs Financiers 2021–2025")
    note = '' if name == 'Google Belgium' else '  ⚠️  Données estimatives'
    print(f"{'=' * 70}{note}")
    display(df_fin[cols].style.format(fmt).set_caption(name))


---
### 3.2 — Diagrammes de Sankey

Visualisation du compte de résultats pour chaque entreprise (année 2025).
Flux : **CA → Marge brute / Coût des ventes → EBITDA → Résultat net / Charges**


In [ ]:
def create_sankey(company_name, fin_df, year=2025, color='#1f77b4'):
    '''
    Génère un diagramme de Sankey pour le compte de résultats d une entreprise.
    Décompose : CA → Marge / Coûts → EBITDA / Charges → Résultat net / Impôts.
    '''
    row        = fin_df.loc[year]
    ca         = float(row['chiffre_affaires'])
    marge      = float(row['marge_brute'])
    ebitda     = float(row['ebitda'])
    rn         = float(row['resultat_net'])

    cout_ventes      = ca    - marge   # Coût des marchandises vendues
    charges_op       = marge - ebitda  # Charges opérationnelles (frais généraux, personnel)
    impots_et_autres = ebitda - rn     # Impôts + charges financières nettes

    def fmt(v):
        '''Formate les montants en M€ avec 1 décimale.'''
        return f"{v/1e6:.1f} M€"

    # ─── Nœuds du graphe ──────────────────────────────────────────────────────
    labels = [
        f"CA\n{fmt(ca)}",                              # 0
        f"Marge brute\n{fmt(marge)}",                  # 1
        f"Coût des ventes\n{fmt(cout_ventes)}",        # 2
        f"EBITDA\n{fmt(ebitda)}",                      # 3
        f"Charges opér.\n{fmt(charges_op)}",           # 4
        f"Résultat net\n{fmt(rn)}",                    # 5
        f"Impôts & charges\n{fmt(impots_et_autres)}",  # 6
    ]
    sources = [0, 0, 1, 1, 3, 3]
    targets = [1, 2, 3, 4, 5, 6]
    values  = [marge, cout_ventes, ebitda, charges_op, rn, impots_et_autres]

    node_colors = [color, '#2ca02c', '#d62728', '#ff7f0e', '#9467bd', '#2ca02c', '#d62728']
    link_colors = [
        'rgba(44,160,44,0.4)', 'rgba(214,39,40,0.4)',
        'rgba(255,127,14,0.4)', 'rgba(148,103,189,0.4)',
        'rgba(44,160,44,0.5)', 'rgba(214,39,40,0.4)',
    ]

    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(
            pad=20, thickness=22,
            line=dict(color='black', width=0.5),
            label=labels, color=node_colors,
        ),
        link=dict(source=sources, target=targets, value=values, color=link_colors),
    ))
    fig.update_layout(
        title_text=f"{company_name} — Sankey du Compte de Résultats {year}",
        title_font_size=15,
        font_size=12,
        height=460,
    )
    fig.show()

# ─── Sankeys pour les 3 entreprises ──────────────────────────────────────────
create_sankey('Google Belgium',       google_fin, year=2025, color='#4285F4')
create_sankey('Apple Retail Belgium', apple_fin,  year=2025, color='#555555')
create_sankey('SNCB',                 sncb_fin,   year=2025, color='#E3001B')


---
### 3.3 — Évolution CA & Résultat Net (2021–2025)

Comparaison de l'évolution financière des 3 entreprises.


In [ ]:
years  = [2021, 2022, 2023, 2024, 2025]
colors = {
    'Google Belgium':       '#4285F4',
    'Apple Retail Belgium': '#555555',
    'SNCB':                 '#E3001B',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Évolution Financière 2021–2025", fontsize=16, fontweight='bold')

# ─── CA ──────────────────────────────────────────────────────────────────────
ax1 = axes[0]
for name, df_fin in companies_fin.items():
    ca_vals = [df_fin.loc[y, 'chiffre_affaires'] / 1e6 for y in years]
    ax1.plot(years, ca_vals, marker='o', linewidth=2.5, markersize=7,
             color=colors[name], label=name)
    ax1.annotate(f"{ca_vals[-1]:.0f} M€",
                 xy=(2025, ca_vals[-1]), xytext=(3, 4),
                 textcoords='offset points', fontsize=9, color=colors[name])
ax1.set_title("Chiffre d'affaires (M€)", fontsize=13)
ax1.set_xlabel("Année")
ax1.set_ylabel("Millions €")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(years)

# ─── Résultat net ─────────────────────────────────────────────────────────────
ax2 = axes[1]
for name, df_fin in companies_fin.items():
    rn_vals = [df_fin.loc[y, 'resultat_net'] / 1e6 for y in years]
    ax2.plot(years, rn_vals, marker='s', linewidth=2.5, markersize=7,
             color=colors[name], label=name)
    ax2.annotate(f"{rn_vals[-1]:.0f} M€",
                 xy=(2025, rn_vals[-1]), xytext=(3, 4),
                 textcoords='offset points', fontsize=9, color=colors[name])
ax2.set_title("Résultat Net (M€)", fontsize=13)
ax2.set_xlabel("Année")
ax2.set_ylabel("Millions €")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(years)

plt.tight_layout()
plt.savefig(f"{DATA_PATH}evolution_financiere.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphique sauvegardé : evolution_financiere.png")


In [ ]:
# ─── Marges comparatives (barres groupées) ───────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
fig.suptitle("Comparaison des Marges Nettes (%) — 2021–2025",
             fontsize=14, fontweight='bold')

bar_w = 0.25
x     = np.arange(len(years))

for i, (name, df_fin) in enumerate(companies_fin.items()):
    margins = [df_fin.loc[y, 'marge_nette_%'] for y in years]
    bars = ax.bar(x + i * bar_w, margins, bar_w,
                  label=name, color=colors[name], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, margins):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.1,
                f"{val:.1f}%", ha='center', va='bottom', fontsize=8)

ax.set_xlabel("Année")
ax.set_ylabel("Marge nette (%)")
ax.set_xticks(x + bar_w)
ax.set_xticklabels(years)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DATA_PATH}marges_comparatives.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphique sauvegardé : marges_comparatives.png")


---
## Glossaire des Indicateurs Financiers

Tous les indicateurs sont calculés d'après le **Plan Comptable Minimum Normalisé (PCMN)** belge.
Les codes entre parenthèses correspondent aux lignes des CSVs téléchargés via la NBB.

---

### Performance

| Indicateur | Définition | Formule | Code(s) PCMN |
|---|---|---|---|
| **Chiffre d'affaires (CA)** | Revenu total généré par la vente de biens et services | — | `70` |
| **Marge brute** | CA après déduction du coût direct des marchandises | `CA − 60` | `70 − 60` |
| **EBIT** | Résultat d'exploitation avant intérêts et impôts | — | `9901` |
| **EBITDA** | EBIT + amortissements — proxy du cash opérationnel | `9901 + 630` | `9901 + 630` |
| **Résultat net** | Bénéfice final après toutes charges, intérêts et impôts | — | `9904` |

---

### Croissance & Marges

| Indicateur | Formule |
|---|---|
| Taux de croissance CA (%) | `(CA_n − CA_{n-1}) / CA_{n-1} × 100` |
| Taux de marge brute (%) | `Marge brute / CA × 100` |
| Marge EBITDA (%) | `EBITDA / CA × 100` |
| Marge nette (%) | `Résultat net / CA × 100` |

---

### Autonomie Financière

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Trésorerie** | Liquidités disponibles (banque + caisse) | — | `54/58` |
| **Dettes financières** | Emprunts bancaires LT + CT | `17 + 43` | `17 + 43` |
| **Dette nette** | Endettement réel après trésorerie | `Dettes fin. − Trésorerie` | — |

---

### Solvabilité

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Fonds propres** | Capital + réserves + résultats reportés | — | `10/15` |
| **Total actif** | Ensemble des ressources de l'entreprise | — | `20/58` |
| **Autonomie financière (%)** | Part financée par fonds propres | `FP / Total actif × 100` | — |

---

### Ressources Humaines

| Indicateur | Formule | Code(s) |
|---|---|---|
| Effectif FTE | Équivalents temps plein moyens | `9087` |
| Coûts salariaux | Rémunérations + charges patronales | `62` |
| Revenu par ETP | `CA / FTE` | — |
| Taxes payées (ISOC) | Impôt des sociétés effectivement payé | `670/3` |
